##### Dataset

In [ ]:

import sys
!{sys.executable} -m pip install liac-arff scikit-multilearn

from skmultilearn.dataset import load_from_arff
import numpy as np

# ---- Train ----
X_train, y_train = load_from_arff(
    "PlantGO-train.arff",
    label_count=12,        # PlantGO has 12 labels
    label_location="end",
    load_sparse=True
)

# ---- Test ----
X_test, y_test = load_from_arff(
    "PlantGO-test.arff",
    label_count=12,
    label_location="end",
    load_sparse=True
)

# Convert sparse → dense (if your SVM needs NumPy)
X_train = X_train.toarray().astype(np.float32)
X_test = X_test.toarray().astype(np.float32)

y_train = y_train.toarray().astype(np.int32)
y_test = y_test.toarray().astype(np.int32)

print("Train X:", X_train.shape)
print("Train y:", y_train.shape)
print("Test X:", X_test.shape)
print("Test y:", y_test.shape)

# Save
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

ModuleNotFoundError: No module named 'arff'

##### SVM Multi Label Algorithm

**Training/Testing Pseudocode**
procedure TRAIN_BINARY_SVM(X, y, learning_rate, lambda_param, n_iters):
    # y is in {0,1}
    y_mapped ← map y to {-1,+1}:
        y_mapped[i] = -1 if y[i] ≤ 0 else +1

    (n_samples, n_features) ← shape(X)
    w ← zero vector of length n_features
    b ← 0

    for iter in 1..n_iters:
        for i in 1..n_samples:
            x_i ← X[i]
            y_i ← y_mapped[i]

            margin_ok ← ( y_i * ( dot(x_i, w) + b ) ) ≥ 1

            if margin_ok:
                # only regularization update
                w ← w - learning_rate * (2 * lambda_param * w)
            else:
                # hinge-loss + regularization update
                w ← w - learning_rate * (2 * lambda_param * w - y_i * x_i)
                b ← b - learning_rate * y_i

    return (w, b)

procedure TRAIN_MULTILABEL_SVM(X, Y, svm_params):
    # Y is binary indicator matrix in {0,1}, shape (n_samples, n_labels)
    L ← number of labels = number of columns in Y
    classifiers ← empty map/dictionary

    for label_idx in 1..L:
        y_label ← Y[:, label_idx]              # one column (binary)
        (w, b) ← TRAIN_BINARY_SVM(X, y_label, svm_params)
        classifiers[label_idx] ← (w, b)

    return classifiers

procedure PREDICT_BINARY_SVM(X, w, b):
    # outputs labels in {0,1}
    scores ← X · w + b                         # vector length n_samples
    y_pred ← 1 where scores ≥ 0 else 0
    return y_pred

procedure PREDICT_MULTILABEL_SVM(X, classifiers):
    L ← number of classifiers
    predictions_list ← empty list

    for label_idx in 1..L:
        (w, b) ← classifiers[label_idx]
        preds_label ← PREDICT_BINARY_SVM(X, w, b)   # shape (n_samples,)
        append preds_label to predictions_list

    Ŷ ← transpose(stack(predictions_list))          # shape (n_samples, L)
    return Ŷ

In [ ]:
class BinarySVM:
    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iters=1000):
        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.w = None
        self.b = None

    def fit(self, X, y):
        # Map labels from {0, 1} to {-1, 1}
        y_mapped = np.where(y <= 0, -1, 1)
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0

        for _ in range(self.n_iters):
            for idx, x_i in enumerate(X):
                condition = y_mapped[idx] * (np.dot(x_i, self.w) + self.b) >= 1
                if condition:
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    self.w -= self.lr * (2 * self.lambda_param * self.w - np.dot(x_i, y_mapped[idx]))
                    self.b -= self.lr * y_mapped[idx]

    def predict(self, X):
        linear_model = np.dot(X, self.w) + self.b
        return np.where(linear_model >= 0, 1, 0) # Map back to {0, 1}
    
class MultiLabelSVM:
    def __init__(self, **svm_params):
        self.svm_params = svm_params
        self.classifiers = {}
        self.labels = None

    def fit(self, X, Y):
        # Y is a binary indicator matrix (n_samples, n_labels)
        self.labels = range(Y.shape[1])
        for label_idx in self.labels:
            # Train a separate BinarySVM for each label
            y_label = Y[:, label_idx]
            svm_classifier = BinarySVM(**self.svm_params)
            svm_classifier.fit(X, y_label)
            self.classifiers[label_idx] = svm_classifier

    def predict(self, X):
        predictions = []
        for label_idx in self.labels:
            # Get predictions from each classifier
            preds = self.classifiers[label_idx].predict(X)
            predictions.append(preds)
        
        # Combine predictions into a binary indicator matrix
        return np.array(predictions).T

    


##### Evaluation Metrics

1. Hamming Loss

Fraction of label decisions that are wrong (averaged over all samples × labels).

Good because it evaluates each label independently (robust when exact match is rare).
2. Micro-averaged Precision / Recall / F1

Aggregates TP/FP/FN across all labels → emphasizes performance on frequent labels and overall correctness.

Best when labels are imbalanced and you care about global performance.

3. Macro-averaged F1

Compute F1 per label, then average → treats each label equally.

Best when you care about rare labels too.

4. Subset Accuracy (Exact Match)

Strict: counts a sample correct only if all labels match.

Useful but often low; include as a “hard” metric to show exact multi-label correctness.

In [ ]:
def multilabel_confusion(Y_true, Y_pred):
    # Y_* shape: (n_samples, n_labels) with {0,1}
    tp = np.sum((Y_true == 1) & (Y_pred == 1))
    fp = np.sum((Y_true == 0) & (Y_pred == 1))
    fn = np.sum((Y_true == 1) & (Y_pred == 0))
    tn = np.sum((Y_true == 0) & (Y_pred == 0))
    return tp, fp, fn, tn

def micro_precision_recall_f1(Y_true, Y_pred, eps=1e-12):
    tp, fp, fn, _ = multilabel_confusion(Y_true, Y_pred)
    prec = tp / (tp + fp + eps)
    rec  = tp / (tp + fn + eps)
    f1   = 2 * prec * rec / (prec + rec + eps)
    return prec, rec, f1

def macro_f1(Y_true, Y_pred, eps=1e-12):
    # average F1 across labels
    L = Y_true.shape[1]
    f1s = []
    for l in range(L):
        yt, yp = Y_true[:, l], Y_pred[:, l]
        tp = np.sum((yt==1) & (yp==1))
        fp = np.sum((yt==0) & (yp==1))
        fn = np.sum((yt==1) & (yp==0))
        prec = tp/(tp+fp+eps)
        rec  = tp/(tp+fn+eps)
        f1   = 2*prec*rec/(prec+rec+eps)
        f1s.append(f1)
    return float(np.mean(f1s))

def hamming_loss(Y_true, Y_pred):
    return float(np.mean(Y_true != Y_pred))

def subset_accuracy(Y_true, Y_pred):
    return float(np.mean(np.all(Y_true == Y_pred, axis=1)))

##### Implementation and Results

In [ ]:
def train_test_report(X_train, Y_train, X_test, Y_test, svm_params):
    model = MultiLabelSVM(**svm_params)
    model.fit(X_train, Y_train)

    Y_pred_train = model.predict(X_train)
    Y_pred_test  = model.predict(X_test)

    def pack_metrics(Yt, Yp):
        p, r, f1 = micro_precision_recall_f1(Yt, Yp)
        return {
            "HammingLoss": hamming_loss(Yt, Yp),
            "SubsetAcc": subset_accuracy(Yt, Yp),
            "MicroP": p, "MicroR": r, "MicroF1": f1,
            "MacroF1": macro_f1(Yt, Yp),
        }

    train_m = pack_metrics(Y_train, Y_pred_train)
    test_m  = pack_metrics(Y_test,  Y_pred_test)

    return train_m, test_m

train_test_report(np.load("X_train.npy"), np.load("y_train.npy"), np.load("X_test.npy"), np.load("y_test.npy"), svm_params={"learning_rate": 0.001, "lambda_param": 0.01, "n_iters": 1000})

##### SVM Multi Class Algorithm

##### Dataset - MNIST: Modified National Institute of Standards and Technology database
It contains: 
    70,000 grayscale images;
    Image size: 28 * 28 pixels;
    10 classes (digits 0-9);
    60,000 training images;
    10,000 test images.

In [8]:
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', version=1)
X = mnist.data
y = mnist.target.astype(int)

print("MNIST X:", X.shape)
print("MNIST y:", y.shape)

MNIST X: (70000, 784)
MNIST y: (70000,)


**Pseudocode**
